## Загрузка датасета SOBHard на HuggingFace

Ноутбук собирает `test.json` и `shots.json` из `datasets/SOBHard/` в `datasets.DatasetDict`
и заливает **две** копии на 🤗 Hub. У обеих по два сплита, `shots` и `test`, с одними и теми же
строками; отличие ровно одно:

* `MERA-evaluation/SOBHard` — по ней гоняется lm-eval. Метрики там не считаются, нужны только
  генерации модели, поэтому в оцениваемых строках пусты и ответ, и все поля, по которым его
  можно было бы восстановить или угадать;
* `MERA-evaluation/SOBHard_gold` — её читает удалённый скорер. Те же строки с заполненными
  полями: эталоны и предвычисленные структуры верификаторов.

Что здесь специфично для этого датасета:

* поле `instruction` в локальных файлах — это **индекс** промпта в `dataset_meta.json["prompts"]`;
  перед заливкой он заменяется на сам текст промпта (общее правило MERA);
* `meta.reference` — эталонный документ, по которому считаются метрики содержания. Скоринг читает
  эталон именно оттуда, а не из `outputs`, — и именно поэтому эталон уезжает в отдельную копию,
  а не остаётся рядом с вопросом;
* `meta.checks` и `meta.task_meta` — JSON-строки с предвычисленными данными для верификаторов;
  у 214 вопросов из 825 поле из `checks` буквально равно эталонному документу, так что это
  такой же ответ, только записанный иначе;
* `meta.dialogue_id` / `turn_index` / `n_turns` — координаты хода в многоходовом вопросе. Их читает
  сэмплер, который собирает разговор, и читает **до** разбора `task_meta`, поэтому они лежат
  отдельными полями. У одноходовых вопросов там `null`;
* сплиты разделены по роли, а не по размеру. Ход разговора работает над результатом предыдущего
  хода, поэтому из одной цепочки оценивается ровно один ход — последний, — а все ходы до него
  лежат в `shots` вместе с ответами и служат этому вопросу контекстом. Поэтому `shots` здесь
  не десять демонстраций, а двести с лишним строк;
* перед заливкой прогоняется проверка целостности: каждый эталон обязан пройти все применимые
  ограничения тем же скорером, которым считается метрика.

In [1]:
import json
import os
import sys

import datasets
from tqdm import tqdm


c:\Users\arorlov\.conda\envs\default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\arorlov\.conda\envs\default\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


### Подготовка данных


#### WARNING!

Если ваш датасет является __ПРИВАТНЫМ__, оставьте `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS` равным `True`.
На ХФ даже приватно не должно лежать датасетов с ответами.

Задание здесь — выдать конкретный документ, и без эталона метрики содержания посчитать нечем.
Раньше это решалось компромиссом: `outputs` стирался, а эталон оставался в `meta.reference` рядом
с вопросом. Компромисса больше нет — вместо него две копии с разными ролями.

В копии, по которой гоняется lm-eval, у оцениваемой строки стираются **значения** полей
`outputs`, `meta.reference`, `meta.reference_sha256`, `meta.task_meta`, `meta.checks`,
`meta.base_id`, `meta.fence_tag` и всех полей `meta.categories`. Именно значения, а не ключи:
схема на Hub одна на оба сплита, и попытка оставить в `test` четыре поля `meta`, а в `shots`
одиннадцать, падает с `DatasetGenerationError`.

Два следствия, оба обязательные к прочтению.

1. **Прогон по этой копии запускается только с `--predict_only`.** Без флага харнесс не падает,
   а молча выдаёт нули: у пустого `fence_tag` не сходится ни `pack.fence_tag_correct`, ни
   `syn.parses_target`, и заведомо верный ответ получает `sample_pass_rate 0.00` — число, которое
   выглядит как результат модели. С флагом метрики становятся `bypass`, генерации пишутся, ложных
   чисел нет. Считать по ним нужно отдельно, по `SOBHard_gold`.
2. **`shots` уезжает наружу вместе с ответами.** Это ответы на неоцениваемые вопросы — десять
   демонстраций и ходы разговоров, которые оцениваемым ходам служат контекстом, — так что ничего
   оцениваемого не утекает. Но по сути публикуется двести примеров правильно оформленного ответа,
   и это осознанное решение, а не побочный эффект.

In [2]:
MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS = True

# Поля оцениваемой строки, значения которых не уезжают в генерирующую копию.
# `checks` здесь не про перестраховку: у 214 вопросов из 825 одно из его полей
# буквально равно эталонному документу, а у остальных — исходный документ, уже
# разобранный в каноническую форму, то есть половина задачи «прочитать чужую
# нотацию» снята.
HIDDEN_META_FIELDS = ("base_id", "fence_tag", "reference", "reference_sha256",
                      "task_meta", "checks")

Пути указаны относительно расположения ноутбука в `datasets/SOBHard/`.

In [3]:
path_to_data = "."
path_to_meta = "."

In [4]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


#### Подгрузка данных


In [5]:
shots = load_json(os.path.join(path_to_data, "shots.json"))["data"]
test = load_json(os.path.join(path_to_data, "test.json"))["data"]
meta = load_json(os.path.join(path_to_meta, "dataset_meta.json"))

print(f"shots: {len(shots)}, test: {len(test)}")


shots: 203, test: 825


In [6]:
prompts = meta["prompts"]
len(prompts)


20

#### Обработка полей датасета

На ХФ загружается датасет, где у КАЖДОГО сэмпла вместо числа в поле `instruction` стоит промпт.
Ячейка идемпотентна.


In [7]:
def resolve_prompts(split):
    for card in split:
        if isinstance(card["instruction"], int):
            card["instruction"] = prompts[card["instruction"]]


resolve_prompts(shots)
resolve_prompts(test)

print(test[0]["instruction"][:400])


Короче, есть задачка. Ниже всё написано — сделай и оформи как просят.

Задача:
{task}

Входные данные:
{input_data}

Формат ответа:
{format}

Инструкция:
{question}


#### Координаты многоходовых вопросов

У одноходовых вопросов полей диалога нет вовсе, а схема на Hub должна быть одинаковой для всех
строк. Проставляем `null` там, где их не было: сэмплер, собирающий диалог, трактует `null` как
«это не ход разговора» и не отдаёт такому вопросу никакой истории.

Ходы разговора теперь лежат в обоих сплитах, и это ровно то, ради чего всё затевалось: в `test`
стоит последний ход каждой цепочки — он и оценивается, — а все ходы до него лежат в `shots`,
откуда сэмплер и собирает историю. Разделение проходит по `turn_index` против `n_turns`.

In [8]:
def normalise_dialogue_fields(split):
    for card in split:
        m = card["meta"]
        for key in ("dialogue_id", "turn_index", "n_turns"):
            m.setdefault(key, None)


normalise_dialogue_fields(shots)
normalise_dialogue_fields(test)

graded_turns = [c for c in test if c["meta"]["dialogue_id"] is not None]
context_turns = [c for c in shots if c["meta"]["dialogue_id"] is not None]
dialogues = {c["meta"]["dialogue_id"] for c in graded_turns + context_turns}

print("разговоров:", len(dialogues),
      "| ходов всего:", len(graded_turns) + len(context_turns))
print("оцениваемых последних ходов в test:", len(graded_turns),
      "| одноходовых вопросов в test:", len(test) - len(graded_turns))
print("ходов-контекста в shots:", len(context_turns),
      "| демонстраций в shots:", len(shots) - len(context_turns))

# Оценивается ровно последний ход цепочки, и ровно один.
assert all(c["meta"]["turn_index"] == c["meta"]["n_turns"] for c in graded_turns)
assert all(c["meta"]["turn_index"] < c["meta"]["n_turns"] for c in context_turns)
assert len(graded_turns) == len(dialogues)

разговоров: 75 | ходов всего: 268
оцениваемых последних ходов в test: 75 | одноходовых вопросов в test: 750
ходов-контекста в shots: 193 | демонстраций в shots: 10


#### Проверка целостности перед заливкой

Подставляем `inputs` в `instruction` — промпт должен собираться без ошибок — и прогоняем эталон
через ту же библиотеку проверок, которой считается метрика. Эталон обязан набрать 1.0.

`balance_score` пропускается: на уровне одного вопроса это пара «клетка, зачёт», и смысл она
приобретает только после агрегации по всему прогону. Она проверяется отдельно, ниже.


In [9]:
sys.path.insert(0, os.path.abspath("../../benchmark_tasks/sobhard"))
import utils as U

bad_prompt, bad_gold = [], []
for card in tqdm(shots + test):
    try:
        card["instruction"].format(**card["inputs"])
    except Exception as exc:
        bad_prompt.append((card["meta"]["id"], exc))
    m = U.process_results(card, [card["outputs"]])
    if any(v != 1.0 for v in U.scalar_metrics(m).values()):
        bad_gold.append((card["meta"]["id"], m))

print("промптов, которые не собираются:", len(bad_prompt))
print("эталонов, не набравших 1.0:", len(bad_gold))
assert not bad_prompt and not bad_gold

# И вторая метрика — на наборе целиком, а не по вопросу.
gold_balance = U.balance_score([U.process_results(c, [c["outputs"]])["balance_score"]
                                for c in test])
print("balance_score эталонного набора:", gold_balance)
assert abs(gold_balance - 1.0) < 1e-9

100%|██████████| 1028/1028 [00:10<00:00, 97.31it/s] 


промптов, которые не собираются: 0
эталонов, не набравших 1.0: 0
balance_score эталонного набора: 1.0


#### Что остаётся видно в каждой копии

Ячейка ничего не меняет: считает по копиям данных. Слева — сколько раз каждое ограничение
применимо к строкам `test` в том виде, в каком они уезжают в скорящую копию; справа — то же
самое по генерирующей. Правая колонка и есть ответ на вопрос, почему прогон по генерирующей
копии обязан идти с `--predict_only`: считать там нечего, и любое число в таблице метрик было
бы неправдой.

In [10]:
import copy
from collections import Counter

# Ячейка ничего не меняет: считает по копиям данных, что видно в каждой из них.
# Применимость меряется на самом эталоне — вопрос ровно в том, сколько ограничений
# остаётся, когда отвечает тот, кто заведомо прав.


def blank_graded(card):
    """Строка `test` в том виде, в каком она уезжает в генерирующую копию."""
    card["outputs"] = ""
    for key in HIDDEN_META_FIELDS:
        card["meta"][key] = ""
    for key in card["meta"]["categories"]:
        card["meta"]["categories"][key] = ""
    return card


def applicable(blank):
    probe = copy.deepcopy(test)
    if blank:
        probe = [blank_graded(card) for card in probe]
    else:
        for card in probe:
            card["outputs"] = ""
    alive = Counter()
    for card, orig in zip(probe, test):
        for ch in U.score_response(card, U.normalize_generation(orig["outputs"])):
            if ch["applicable"]:
                alive[ch["id"]] += 1
    return alive


gold_side, generating_side = applicable(False), applicable(True)

print(f"{'ограничение':34s} {'SOBHard_gold':>14s} {'SOBHard':>10s}")
for cid in U.constraint_ids():
    print(f"{cid:34s} {gold_side.get(cid, 0):>14d} {generating_side.get(cid, 0):>10d}")
print(f"{'ИТОГО применений':34s} {sum(gold_side.values()):>14d} "
      f"{sum(generating_side.values()):>10d}")

# И что при этом происходит с метриками эталона.
for name, blank in (("SOBHard_gold", False), ("SOBHard", True)):
    probe = copy.deepcopy(test)
    if blank:
        probe = [blank_graded(card) for card in probe]
    else:
        for card in probe:
            card["outputs"] = ""
    n_bad = sum(any(v != 1.0 for v in U.scalar_metrics(
        U.process_results(card, [orig["outputs"]])).values())
        for card, orig in zip(probe, test))
    print(f"{name}: эталонов не на 1.0 — {n_bad} из {len(test)}")

ограничение                          SOBHard_gold    SOBHard
pack.fence_present                            825        825
pack.fence_exactly_one                        825        825
pack.fence_tag_correct                        825        825
pack.nothing_outside_fence                    825        825
syn.parses_target                             825        825
val.structure_match                           825          0
val.numeric_within_tol                        825          0
val.strings_byte_exact                        825          0
convert.lossless_vs_source                     75          0
extract.keys_exact                             75          0
extract.is_object                              75          0
extract.absent_paths_null                      75          0
repair.data_intact                             75          0
transform.rule_applied                        297          0
transform.list_order_preserved                297          0
transform.values_unchang

#### Разделяем копии по ролям

`gold` — то, что уедет в скорящую копию: эталон и всё, из чего верификаторы выводят ответ.
`test` — то, что уедет в генерирующую: у оцениваемой строки остаются только промпт, `id` и
координаты хода в разговоре, по которым сэмплер найдёт историю в `shots`.

В `shots` ничего не стирается. Это неоцениваемые вопросы: десять демонстраций и ходы, ответы
которых и есть история оцениваемых последних ходов, — без ответа они бесполезны.

In [11]:
# Скорящая копия — снимок до стирания. `outputs` пуст и там: скорер читает эталон
# из meta.reference, а держать один и тот же ответ в двух полях незачем.
gold = copy.deepcopy(test)
for card in gold:
    card["outputs"] = ""

if MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS:
    for card in tqdm(test):
        blank_graded(card)

print("SOBHard      — оцениваемых строк:", len(test))
print("  outputs стёрт:", all(c["outputs"] == "" for c in test))
print("  meta стёрта:", all(not c["meta"][k] for c in test
                            for k in HIDDEN_META_FIELDS))
print("  categories стёрты:", all(not v for c in test
                                  for v in c["meta"]["categories"].values()))
print("  координаты хода на месте:",
      sum(c["meta"]["dialogue_id"] is not None for c in test), "строк")
print("SOBHard_gold — оцениваемых строк:", len(gold))
print("  meta.reference на месте:", all(c["meta"]["reference"] for c in gold))
print("shots        — строк:", len(shots),
      "| с непустым outputs:", sum(bool(c["outputs"]) for c in shots))

# Главная проверка разделения: ни один эталон оцениваемого вопроса не встречается
# нигде в shots — ни в ответе, ни в эталоне, ни в тексте промпта.
haystack = "\n".join([c["outputs"] for c in shots]
                     + [c["meta"]["reference"] for c in shots]
                     + [t for c in shots for t in c["inputs"].values()])
leaked = [c["meta"]["id"] for c in gold if c["meta"]["reference"] in haystack]
assert not leaked, leaked[:10]
print("эталонов оцениваемых вопросов, найденных в shots:", len(leaked))

100%|██████████| 825/825 [00:00<00:00, 270230.44it/s]


SOBHard      — оцениваемых строк: 825
  outputs стёрт: True
  meta стёрта: True
  categories стёрты: True
  координаты хода на месте: 75 строк
SOBHard_gold — оцениваемых строк: 825
  meta.reference на месте: True
shots        — строк: 203 | с непустым outputs: 203
эталонов оцениваемых вопросов, найденных в shots: 0


### Создаем датасет для загрузки на ХФ


#### Аннотация полей датасета

* схема одна на оба сплита и на обе копии. Билдер выводит признаки один раз и применяет их ко
  всему, что ему дали, поэтому спрятать поле в одном сплите и оставить в другом нельзя — на
  `push_to_hub` это падает с `DatasetGenerationError`. Прячутся значения, ключи остаются;
* `meta.task_meta` и `meta.checks` — именно **строки** с JSON внутри: у разных семейств разный
  набор параметров, фиксированной схемой их не описать;
* `meta.reference` — сырой текст эталонного документа, а не разобранный JSON. Хранить разобранным
  нельзя: в YAML и TOML ключи отображений бывают не строками, и JSON-сериализация превратила бы их
  в строки только с одной стороны сравнения;
* поля диалога объявлены как целые и допускают `null` — у одноходовых вопросов их нет. В
  генерирующей копии они остаются заполненными и у оцениваемых строк: по ним сэмплер находит в
  `shots` предыдущие ходы разговора.

In [12]:
features = datasets.Features({
    "instruction": datasets.Value("string"),
    "inputs": {
        "task": datasets.Value("string"),
        "input_data": datasets.Value("string"),
        "format": datasets.Value("string"),
        "question": datasets.Value("string"),
    },
    "outputs": datasets.Value("string"),
    "meta": {
        "id": datasets.Value("int32"),
        "base_id": datasets.Value("string"),
        "fence_tag": datasets.Value("string"),
        "reference": datasets.Value("string"),
        "reference_sha256": datasets.Value("string"),
        "task_meta": datasets.Value("string"),
        "checks": datasets.Value("string"),
        # Координаты хода в разговоре; null у одноходовых вопросов.
        "dialogue_id": datasets.Value("int32"),
        "turn_index": datasets.Value("int32"),
        "n_turns": datasets.Value("int32"),
        "categories": {
            "family": datasets.Value("string"),
            "difficulty": datasets.Value("string"),
            "language": datasets.Value("string"),
            "source_format": datasets.Value("string"),
            "target_format": datasets.Value("string"),
            "length_tier": datasets.Value("string"),
            "prompt_style": datasets.Value("string"),
            # Происхождение: из какого файла корпуса взят документ.
            "origin": datasets.Value("string"),
        },
    },
})


#### Создание датасетов для каждого сплита


In [13]:
shots_ds = datasets.Dataset.from_list(shots, features=features)
test_ds = datasets.Dataset.from_list(test, features=features)
gold_ds = datasets.Dataset.from_list(gold, features=features)
shots_ds, test_ds, gold_ds

(Dataset({
     features: ['instruction', 'inputs', 'outputs', 'meta'],
     num_rows: 203
 }),
 Dataset({
     features: ['instruction', 'inputs', 'outputs', 'meta'],
     num_rows: 825
 }),
 Dataset({
     features: ['instruction', 'inputs', 'outputs', 'meta'],
     num_rows: 825
 }))

##### Проверка

Ничего не потеряно, не продублировано и не переехало не туда — включая порядок ходов в разговорах
и правило, по которому они разложены на два сплита.

In [14]:
from collections import Counter

# количество вопросов до конвертации и после совпадает
assert len(test) == len(test_ds) and len(shots) == len(shots_ds)
assert len(gold) == len(gold_ds)

# id вопросов сходятся и остаются сквозными по объединению сплитов. id — ключ,
# по которому удалённый скорер сопоставляет ответы модели с эталонами, так что
# он обязан быть уникальным; после разделения по ролям нумерация проходит
# насквозь: сначала shots, потом test.
assert [c["meta"]["id"] for c in test] == [c["meta"]["id"] for c in test_ds]
assert [c["meta"]["id"] for c in shots] == [c["meta"]["id"] for c in shots_ds]
assert [c["meta"]["id"] for c in gold] == [c["meta"]["id"] for c in test_ds]
ids = sorted(c["meta"]["id"] for c in shots + test)
assert ids == list(range(1, len(ids) + 1))

# сетка: в каждой клетке ровно 25 вопросов. Клетка читается по скорящей копии —
# в генерирующей категории стёрты вместе с остальными подсказками.
cells = Counter((c["meta"]["categories"]["family"],
                 c["meta"]["categories"]["difficulty"]) for c in gold_ds)
assert set(cells.values()) == {25}, cells
print("клеток:", len(cells), "| вопросов в клетке: 25")

# meta.checks и meta.task_meta всё ещё парсятся — там, где они остались
assert all(json.loads(c["meta"]["checks"]) is not None for c in gold_ds)
assert all(json.loads(c["meta"]["task_meta"]) is not None for c in gold_ds)
assert all(c["meta"]["checks"] == "" for c in test_ds)

# разговоры пережили конвертацию: ходы идут по порядку, без дыр, и разложены по
# двум сплитам по одному правилу — последний ход в test, все предыдущие в shots
by_dialogue = {}
for c in list(shots_ds) + list(test_ds):
    did = c["meta"]["dialogue_id"]
    if did is not None:
        by_dialogue.setdefault(did, []).append(c["meta"])
for did, turns in by_dialogue.items():
    idx = sorted(t["turn_index"] for t in turns)
    assert idx == list(range(1, len(idx) + 1)), (did, idx)
    assert {t["n_turns"] for t in turns} == {len(idx)}, (did, turns)
graded_ids = {c["meta"]["dialogue_id"] for c in test_ds
              if c["meta"]["dialogue_id"] is not None}
assert graded_ids == set(by_dialogue), "у разговора нет оцениваемого хода"
assert all(c["meta"]["turn_index"] == c["meta"]["n_turns"] for c in test_ds
           if c["meta"]["dialogue_id"] is not None), \
    "в test попал не последний ход — его ответ пришлось бы показать"
assert any(c["meta"]["dialogue_id"] is not None for c in shots_ds), \
    "в shots нет ходов разговора — оцениваемым последним ходам не из чего собрать историю"
print("разговоров:", len(by_dialogue), "| ходов:", sum(len(v) for v in by_dialogue.values()))
print("OK")

клеток: 33 | вопросов в клетке: 25
разговоров: 75 | ходов: 268
OK


#### Собираем сплиты в датасеты

У обеих копий по два сплита, `shots` и `test`, с одними и теми же строками в одном и том же
порядке. Отличие между копиями ровно одно — в генерирующей часть полей оцениваемой строки пуста.
Одинаковые сплиты нужны не для симметрии: прогон по любой из копий собирает историю разговора
из `shots`, и копия без него мерила бы не то.

In [15]:
dataset = datasets.DatasetDict({"shots": shots_ds, "test": test_ds})
gold_dataset = datasets.DatasetDict({"shots": shots_ds, "test": gold_ds})
dataset, gold_dataset

(DatasetDict({
     shots: Dataset({
         features: ['instruction', 'inputs', 'outputs', 'meta'],
         num_rows: 203
     })
     test: Dataset({
         features: ['instruction', 'inputs', 'outputs', 'meta'],
         num_rows: 825
     })
 }),
 DatasetDict({
     shots: Dataset({
         features: ['instruction', 'inputs', 'outputs', 'meta'],
         num_rows: 203
     })
     test: Dataset({
         features: ['instruction', 'inputs', 'outputs', 'meta'],
         num_rows: 825
     })
 }))

### Загрузка датасета на ХФ

Понадобятся токен с правом записи и путь для записи. Название пишите ровно так, как оно заявлено
в мете (`dataset_meta.json["dataset_name"]`), регистр имеет значение.

Советуем сначала залить приватно и выслать на mera@a-ai.ru токен и путь для верификации.


In [ ]:
from dotenv import load_dotenv

load_dotenv('../../.env')

### TOKEN
token = os.getenv('HF_TOKEN')
if token is None:
    raise ValueError("HF_TOKEN not found in .env file")

### UPLOAD PATH
HF_REPO_ID = "MERA-evaluation/SOBHard"
# Скорящая копия. Отдельный репозиторий, а не отдельный сплит того же датасета:
# сплит скачивался бы вместе с остальными, и разделение было бы декоративным.
HF_GOLD_REPO_ID = "MERA-evaluation/SOBHard_gold"

# Предварительный просмотр — залейте в свой приватный репозиторий:
# HF_REPO_ID = "<your-account>/SOBHard"
# HF_GOLD_REPO_ID = "<your-account>/SOBHard_gold"

### PRIVATE OR PUBLIC
upload_private = True

print("upload to:", HF_REPO_ID, "and", HF_GOLD_REPO_ID,
      "| private:", upload_private)

upload to: MERA-evaluation/SOBHard and MERA-evaluation/SOBHard_gold | private: True


In [19]:
dataset.push_to_hub(HF_REPO_ID, private=upload_private, token=token)
gold_dataset.push_to_hub(HF_GOLD_REPO_ID, private=True, token=token)

Setting num_proc from 1 back to 1 for the shots split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 75.69ba/s]
Processing Files (1 / 1): 100%|██████████| 6.43MB / 6.43MB, 1.89MB/s  
New Data Upload: 100%|██████████| 6.36MB / 6.36MB, 1.87MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.21s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 83.91ba/s]
Processing Files (1 / 1): 100%|██████████| 14.0MB / 14.0MB, 10.0MB/s  
New Data Upload: 100%|██████████| 7.79MB / 7.79MB, 5.56MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.39s/ shards]
Setting num_proc from 1 back to 1 for the shots split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 98

CommitInfo(commit_url='https://huggingface.co/datasets/MERA-evaluation/SOBHard_gold/commit/097c7b27b286a95abaded1726583622c1bcc43ac', commit_message='Upload dataset', commit_description='', oid='097c7b27b286a95abaded1726583622c1bcc43ac', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MERA-evaluation/SOBHard_gold', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MERA-evaluation/SOBHard_gold'), pr_revision=None, pr_num=None)

### Проверка того, как датасет загрузился на ХФ

Загрузим датасет обратно и убедимся, что его увидит корректно любой, кто его скачает.


In [20]:
reloaded = datasets.load_dataset(HF_REPO_ID, token=token)
reloaded_gold = datasets.load_dataset(HF_GOLD_REPO_ID, token=token)
reloaded, reloaded_gold

c:\Users\arorlov\.conda\envs\default\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arorlov\.cache\huggingface\hub\datasets--MERA-evaluation--SOBHard. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 825/825 [00:00<00:00, 30819.32 examples/s]
c:\Users\arorlov\.con

(DatasetDict({
     shots: Dataset({
         features: ['instruction', 'inputs', 'outputs', 'meta'],
         num_rows: 203
     })
     test: Dataset({
         features: ['instruction', 'inputs', 'outputs', 'meta'],
         num_rows: 825
     })
 }),
 DatasetDict({
     shots: Dataset({
         features: ['instruction', 'inputs', 'outputs', 'meta'],
         num_rows: 203
     })
     test: Dataset({
         features: ['instruction', 'inputs', 'outputs', 'meta'],
         num_rows: 825
     })
 }))

In [21]:
# сплиты, размеры и сетка совпадают с локальными
assert len(reloaded["test"]) == len(test) and len(reloaded["shots"]) == len(shots)
assert len(reloaded_gold["test"]) == len(gold)
assert len(reloaded_gold["shots"]) == len(shots)
back = Counter((c["meta"]["categories"]["family"],
                c["meta"]["categories"]["difficulty"])
               for c in reloaded_gold["test"])
assert back == cells

# разговоры доехали целыми и остались разложены по ролям
back_graded = [c for c in reloaded["test"] if c["meta"]["dialogue_id"] is not None]
back_context = [c for c in reloaded["shots"] if c["meta"]["dialogue_id"] is not None]
assert len(back_graded) + len(back_context) == sum(len(v) for v in by_dialogue.values())
assert all(c["meta"]["turn_index"] == c["meta"]["n_turns"] for c in back_graded)

# в генерирующей копии не осталось ни ответа, ни подсказки к нему
assert all(not c["outputs"] for c in reloaded["test"])
assert all(not c["meta"][k] for c in reloaded["test"] for k in HIDDEN_META_FIELDS)
assert all(not v for c in reloaded["test"] for v in c["meta"]["categories"].values())

# отличие между копиями — ровно эти поля и ничего больше
assert ([c["instruction"] for c in reloaded["test"]]
        == [c["instruction"] for c in reloaded_gold["test"]])
assert ([c["inputs"] for c in reloaded["test"]]
        == [c["inputs"] for c in reloaded_gold["test"]])
assert ([c["meta"]["id"] for c in reloaded["test"]]
        == [c["meta"]["id"] for c in reloaded_gold["test"]])

# и ни один эталон не читается в shots
back_haystack = "\n".join([c["outputs"] for c in reloaded["shots"]]
                          + [c["meta"]["reference"] for c in reloaded["shots"]]
                          + [t for c in reloaded["shots"]
                             for t in c["inputs"].values()])
assert not [c["meta"]["id"] for c in reloaded_gold["test"]
            if c["meta"]["reference"] in back_haystack]

# эталон не подменился по дороге: он тот, который произвела сборка
import hashlib
assert all(hashlib.sha256(c["meta"]["reference"].encode("utf-8")).hexdigest()
           == c["meta"]["reference_sha256"] for c in reloaded_gold["test"])

# и скорер по-прежнему ставит эталону 1.0 — на всём сплите, а не на выборке
rows = list(reloaded_gold["test"])
for card in tqdm(rows):
    gold_answer = f"```{card['meta']['fence_tag']}\n{card['meta']['reference']}\n```"
    m = U.scalar_metrics(U.process_results(card, [gold_answer]))
    assert all(v == 1.0 for v in m.values()), (card["meta"]["id"], m)
bs = U.balance_score([U.process_results(
    c, [f"```{c['meta']['fence_tag']}\n{c['meta']['reference']}\n```"])["balance_score"]
    for c in rows])
assert abs(bs - 1.0) < 1e-9, bs
print(f"эталон в SOBHard_gold набирает 1.0 на всех {len(rows)} вопросах, "
      f"balance_score = {bs}")
print("OK")

100%|██████████| 825/825 [00:08<00:00, 94.99it/s] 


эталон в SOBHard_gold набирает 1.0 на всех 825 вопросах, balance_score = 1.0
OK
